# 2.3 — Gradient Descent

Gradient descent turns local slope information into repeated improvement: compute the gradient, step in the opposite direction, and let many small moves replace an impossible exact solve of $\nabla f(x)=0$. This lesson builds the update from scratch, shows why the learning rate matters, and connects the same idea to regression, feature scaling, and convergence diagnostics.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build gradient descent one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so the update is never a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, vector arithmetic, and small numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for random starting points.

### 1. The gradient is the local uphill direction

Gradient descent starts with one local fact: the derivative tells us which way the function increases fastest. For a one-dimensional function $f(x)=(x-3)^2$, the derivative is $f'(x)=2(x-3)$. If $x=0$, the derivative is negative, which means increasing $x$ moves downhill even though the derivative itself points uphill toward decreasing $x$.

In [ ]:
x_grid_w = np.linspace(-1, 7, 200)  # points for drawing the quadratic bowl.
f_grid_w = (x_grid_w - 3) ** 2  # objective values on the grid.
x0_w = 0.0  # current location.
g0_w = 2 * (x0_w - 3)  # derivative of (x-3)^2 at x0.
print("x0:", x0_w, "gradient:", g0_w, "negative gradient:", -g0_w)
assert g0_w == -6.0

▶ What you'll see: at `x0=0`, the gradient is `-6`, so the descent direction is `+6`.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.plot(x_grid_w, f_grid_w, color="navy")
plt.scatter([x0_w], [(x0_w - 3) ** 2], color="crimson", zorder=3)
plt.arrow(x0_w, 9.0, -0.7, 0, head_width=0.45, color="crimson", length_includes_head=True, label="gradient")
plt.arrow(x0_w, 7.6, 0.7, 0, head_width=0.45, color="seagreen", length_includes_head=True, label="-gradient")
plt.title("1: gradient points uphill; -gradient descends")
plt.xlabel("x"); plt.ylabel("f(x)"); plt.legend(); plt.show()

▶ What you'll see: the red arrow points left/uphill while the green arrow points right/downhill toward the minimizer.

*Why it's done this way:* the first-order approximation says $f(x+s)\approx f(x)+f'(x)s$. To make that local model smaller, choose $s$ with the opposite sign of $f'(x)$; in many dimensions, the same dot-product logic says $s=-\eta\nabla f(x)$ gives immediate predicted decrease.

### 2. One update: direction and step size are separate

The gradient gives the direction, but the learning rate $\eta$ decides how much we trust that local direction. The update is $x_{t+1}=x_t-\eta\nabla f(x_t)$. On the same quadratic, $x_0=0$ and $\eta=0.1$ produce $x_1=0.6$.

In [ ]:
eta_w = 0.1  # step size: how boldly to trust the local slope.
x1_w = x0_w - eta_w * g0_w  # gradient descent update.
f0_w = (x0_w - 3) ** 2
f1_w = (x1_w - 3) ** 2
print("x1:", x1_w, "f0:", f0_w, "f1:", f1_w)
assert round(x1_w, 3) == 0.6 and round(f1_w, 2) == 5.76

▶ What you'll see: the objective drops from `9.0` to `5.76` after one small step.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.plot(x_grid_w, f_grid_w, color="navy")
plt.scatter([x0_w, x1_w], [f0_w, f1_w], color=["crimson", "seagreen"], zorder=3)
plt.plot([x0_w, x1_w], [f0_w, f1_w], "--", color="gray")
plt.title("2: one gradient-descent update")
plt.xlabel("x"); plt.ylabel("f(x)"); plt.show()

▶ What you'll see: the new point moves rightward and downward along the bowl.

*Why it's done this way:* multiplying by $\eta$ separates two decisions that are often confused: the gradient determines the locally best direction, while $\eta$ limits distance because the tangent/linear approximation is only trustworthy nearby.

### 3. Repeating updates accumulates local progress

Gradient descent is useful because one local improvement can be repeated. For $f(x)=(x-3)^2$, the error $e_t=x_t-3$ follows $e_{t+1}=(1-2\eta)e_t$. With $\eta=0.2$, the error multiplies by $0.6$ each step, so the sequence contracts toward the minimizer.

In [ ]:
eta_stable_w = 0.2
xs_w = [0.0]
for t_w in range(20):
    grad_w = 2 * (xs_w[-1] - 3)
    xs_w.append(xs_w[-1] - eta_stable_w * grad_w)
xs_w = np.array(xs_w)
losses_w = (xs_w - 3) ** 2
print("first five x:", np.round(xs_w[:5], 3))
print("final error magnitude:", abs(xs_w[-1] - 3))
assert abs(abs(xs_w[-1] - 3) - 3 * 0.6 ** 20) < 1e-10

▶ What you'll see: `x` moves 0 → 1.2 → 1.92 → 2.352 → ... and the error shrinks geometrically.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(losses_w, marker="o", color="purple")
plt.yscale("log")
plt.title("3: stable steps make loss fall geometrically")
plt.xlabel("iteration"); plt.ylabel("loss, log scale"); plt.show()

▶ What you'll see: the loss curve is nearly a straight descending line on a log scale, the signature of geometric contraction.

*Why it's done this way:* for a quadratic, the update is an exact linear recurrence, so convergence is controlled by $|1-2\eta|<1$. Repetition works because every step keeps shrinking the distance to the optimum instead of merely making one lucky move.

### 4. Step size must respect curvature

The same descent direction can converge or explode depending on $\eta$. For $f(x)=(x-3)^2$, curvature is $L=2$, and constant-step gradient descent is stable for $0<\eta<1$. With $\eta=1.1$, the error multiplier is $-1.2$: the sign flips and the magnitude grows.

In [ ]:
eta_bad_w = 1.1
xs_bad_w = [0.0]
for t_w in range(8):
    grad_bad_w = 2 * (xs_bad_w[-1] - 3)
    xs_bad_w.append(xs_bad_w[-1] - eta_bad_w * grad_bad_w)
xs_bad_w = np.array(xs_bad_w)
print("unstable x path:", np.round(xs_bad_w, 3))
print("final error magnitude:", round(abs(xs_bad_w[-1] - 3), 3))
assert round(abs(xs_bad_w[-1] - 3), 3) == 12.899

▶ What you'll see: the iterates jump across the minimizer with growing distance.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot((xs_w - 3) ** 2, marker="o", label="η=0.2 stable")
plt.plot((xs_bad_w - 3) ** 2, marker="s", label="η=1.1 unstable")
plt.yscale("log")
plt.title("4: learning rate controls stability")
plt.xlabel("iteration"); plt.ylabel("loss, log scale"); plt.legend(); plt.show()

▶ What you'll see: the stable curve falls while the large-step curve rises after oscillating.

*Why it's done this way:* curvature measures how quickly the gradient changes. A large $\eta$ trusts the old gradient too far; once the step overshoots the bowl enough that $|1-\eta L|>1$, local descent turns into global divergence.

### 5. Vector gradients have the same shape as parameters

In more than one dimension, the gradient is a vector of partial derivatives. For $f(w)= (w_0-1)^2+4(w_1+2)^2$, the second coordinate has larger curvature, so the gradient component for $w_1$ can be much larger. The update still subtracts a same-shaped gradient vector.

In [ ]:
w_w = np.array([-2.0, 1.0])
grad_vec_w = np.array([2 * (w_w[0] - 1), 8 * (w_w[1] + 2)])
eta_vec_w = 0.1
w_next_w = w_w - eta_vec_w * grad_vec_w
print("w:", w_w, "gradient:", grad_vec_w, "w_next:", w_next_w)
assert np.allclose(w_next_w, [-1.4, -1.4])

▶ What you'll see: the gradient has two entries, one per parameter, and the steeper coordinate moves more.

In [ ]:
xv_w = np.linspace(-3, 3, 80)
yv_w = np.linspace(-3.5, 2, 80)
Xv_w, Yv_w = np.meshgrid(xv_w, yv_w)
Zv_w = (Xv_w - 1) ** 2 + 4 * (Yv_w + 2) ** 2
plt.figure(figsize=(4.8, 3.6))
plt.contour(Xv_w, Yv_w, Zv_w, levels=18, cmap="viridis")
plt.scatter([w_w[0], w_next_w[0], 1], [w_w[1], w_next_w[1], -2], color=["crimson", "seagreen", "black"])
plt.title("5: vector step on an elliptical bowl")
plt.xlabel("w0"); plt.ylabel("w1"); plt.show()

▶ What you'll see: contours are stretched ellipses, and the update moves sharply in the high-curvature coordinate.

*Why it's done this way:* each partial derivative answers “if I change only this coordinate, how fast does the objective change?” Stacking them gives the direction of steepest increase under Euclidean distance, so subtracting that vector is the most direct local decrease step.

### 6. Regression gradients average evidence from all examples

For linear regression with prediction $\hat y_i=w x_i$ and loss $L(w)=\frac{1}{n}\sum_i (w x_i-y_i)^2$, the derivative is $\frac{1}{n}\sum_i 2x_i(w x_i-y_i)$. At $w=0$ for $x=(1,2,3)$ and $y=2x$, every example says the slope is too small, so the gradient is strongly negative.

In [ ]:
X_reg_w = np.array([1.0, 2.0, 3.0])
y_reg_w = 2 * X_reg_w
w0_reg_w = 0.0
resid_reg_w = w0_reg_w * X_reg_w - y_reg_w
terms_reg_w = 2 * X_reg_w * resid_reg_w
grad_reg_w = terms_reg_w.mean()
print("gradient terms:", terms_reg_w, "mean gradient:", round(grad_reg_w, 3))
assert round(grad_reg_w, 3) == -18.667

▶ What you'll see: terms `[-4, -16, -36]` average to `-18.667`.

In [ ]:
eta_reg_w = 0.05
w1_reg_w = w0_reg_w - eta_reg_w * grad_reg_w
loss0_reg_w = np.mean((w0_reg_w * X_reg_w - y_reg_w) ** 2)
loss1_reg_w = np.mean((w1_reg_w * X_reg_w - y_reg_w) ** 2)
print("w1:", round(w1_reg_w, 3), "loss before/after:", round(loss0_reg_w, 3), round(loss1_reg_w, 3))
assert round(w1_reg_w, 3) == 0.933

▶ What you'll see: the first regression step jumps the slope from `0` to about `0.933` and lowers MSE.

In [ ]:
plt.figure(figsize=(4.8, 3.2))
plt.scatter(X_reg_w, y_reg_w, color="black", label="data")
plt.plot(X_reg_w, w0_reg_w * X_reg_w, "--", label="before")
plt.plot(X_reg_w, w1_reg_w * X_reg_w, label="after one step")
plt.title("6: regression gradient moves the line toward data")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

▶ What you'll see: the fitted line rotates upward because all residuals agreed the slope was too low.

*Why it's done this way:* the gradient is an average of per-example forces, so examples with larger $x_i$ and larger residuals contribute more. The update is large here because every term points in the same direction: increase $w$.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per mechanic in this lesson. Each uses a handful of small
> numbers, prints every intermediate value with an inline `# ->` showing the result, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · The gradient points uphill

The derivative is the local uphill direction. A negative derivative means the negative-gradient
direction points to the right, where this bowl decreases.

In [ ]:
import numpy as np                              # arrays and derivative checks.
import matplotlib.pyplot as plt                 # one picture per toy.

t1_rng = np.random.default_rng(0)               # -> seed 0
print("rng seed:", 0)                           # -> 0
t1_x = 1.0                                      # -> 1.0
print("current x:", t1_x)                       # -> 1.0
t1_minimizer = 4.0                              # -> 4.0
print("minimizer:", t1_minimizer)               # -> 4.0
t1_grad = 2.0 * (t1_x - t1_minimizer)           # -> -6.0
print("gradient:", t1_grad)                     # -> -6.0
t1_descent = -t1_grad                           # -> 6.0
print("negative gradient:", t1_descent)         # -> 6.0
t1_probe = np.array([-0.5, 0.5])                # -> [-0.5, 0.5]
print("probe moves:", t1_probe.tolist())        # -> [-0.5, 0.5]
t1_linear = t1_grad * t1_probe                  # -> [3.0, -3.0]
print("linearized changes:", t1_linear.tolist()) # -> [3.0, -3.0]
t1_grid = np.linspace(0.0, 6.0, 7)              # -> [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
print("plot grid:", t1_grid.tolist())           # -> [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
t1_values = (t1_grid - t1_minimizer) ** 2       # -> [16.0, 9.0, 4.0, 1.0, 0.0, 1.0, 4.0]
print("f(grid):", t1_values.tolist())           # -> [16.0, 9.0, 4.0, 1.0, 0.0, 1.0, 4.0]
assert t1_grad == -6.0 and t1_linear[1] < 0.0

plt.figure(figsize=(4.6, 3.0))
plt.plot(t1_grid, t1_values, marker="o", color="navy")
plt.scatter([t1_x], [(t1_x - t1_minimizer) ** 2], color="crimson", zorder=3)
plt.arrow(t1_x, 8.0, -0.6, 0.0, head_width=0.35, color="crimson", length_includes_head=True, label="gradient")
plt.arrow(t1_x, 7.0, 0.6, 0.0, head_width=0.35, color="seagreen", length_includes_head=True, label="-gradient")
plt.title("Toy 1 · uphill vs descent")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.show()

▶ What you'll see: the red gradient arrow points left/uphill, while the green descent arrow points right/downhill.

### ✍️ Toy 2 · One update separates direction and step size

The gradient chooses the direction, and the learning rate scales it into a finite move. One small step
can lower the objective without solving the problem exactly.

In [ ]:
import numpy as np                              # scalar updates and checks.

t2_rng = np.random.default_rng(0)               # -> seed 0
print("rng seed:", 0)                           # -> 0
t2_x0 = 1.0                                     # -> 1.0
print("x0:", t2_x0)                             # -> 1.0
t2_minimizer = 4.0                              # -> 4.0
print("minimizer:", t2_minimizer)               # -> 4.0
t2_grad = 2.0 * (t2_x0 - t2_minimizer)          # -> -6.0
print("gradient:", t2_grad)                     # -> -6.0
t2_eta = 0.25                                   # -> 0.25
print("eta:", t2_eta)                           # -> 0.25
t2_step = -t2_eta * t2_grad                     # -> 1.5
print("step:", t2_step)                         # -> 1.5
t2_x1 = t2_x0 + t2_step                         # -> 2.5
print("x1:", t2_x1)                             # -> 2.5
t2_f0 = (t2_x0 - t2_minimizer) ** 2             # -> 9.0
print("f0:", t2_f0)                             # -> 9.0
t2_f1 = (t2_x1 - t2_minimizer) ** 2             # -> 2.25
print("f1:", t2_f1)                             # -> 2.25
t2_drop = t2_f0 - t2_f1                         # -> 6.75
print("decrease:", t2_drop)                     # -> 6.75
t2_grid = np.linspace(0.0, 5.0, 6)              # -> [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]
print("plot grid:", t2_grid.tolist())           # -> [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]
t2_values = (t2_grid - t2_minimizer) ** 2       # -> [16.0, 9.0, 4.0, 1.0, 0.0, 1.0]
print("f(grid):", t2_values.tolist())           # -> [16.0, 9.0, 4.0, 1.0, 0.0, 1.0]
assert t2_x1 == 2.5 and t2_f1 < t2_f0

plt.figure(figsize=(4.6, 3.0))
plt.plot(t2_grid, t2_values, marker="o", color="navy")
plt.scatter([t2_x0, t2_x1], [t2_f0, t2_f1], color=["crimson", "seagreen"], zorder=3)
plt.plot([t2_x0, t2_x1], [t2_f0, t2_f1], color="gray", linestyle="--")
plt.title("Toy 2 · one scaled update")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.show()

▶ What you'll see: the step moves from `1.0` to `2.5` and drops the loss from `9.0` to `2.25`.

### ✍️ Toy 3 · Repeated updates contract the error

On a quadratic with a stable learning rate, each update multiplies the error by the same factor. The
loss shrinks geometrically.

In [ ]:
import numpy as np                              # arrays and repeated updates.

t3_rng = np.random.default_rng(0)               # -> seed 0
print("rng seed:", 0)                           # -> 0
t3_eta = 0.2                                    # -> 0.2
print("eta:", t3_eta)                           # -> 0.2
t3_factor = 1.0 - 2.0 * t3_eta                  # -> 0.6
print("error factor:", t3_factor)               # -> 0.6
t3_xs = np.array([0.0, 1.2, 1.92, 2.352, 2.6112, 2.76672, 2.860032])  # -> six updates from x0=0
print("x path:", np.round(t3_xs, 6).tolist())   # -> [0.0, 1.2, 1.92, 2.352, 2.6112, 2.76672, 2.860032]
t3_errors = t3_xs - 3.0                         # -> [-3.0, -1.8, -1.08, -0.648, -0.3888, -0.23328, -0.139968]
print("errors:", np.round(t3_errors, 6).tolist()) # -> [-3.0, -1.8, -1.08, -0.648, -0.3888, -0.23328, -0.139968]
t3_losses = t3_errors ** 2                      # -> [9.0, 3.24, 1.1664, 0.419904, 0.151165, 0.05442, 0.019591]
print("losses:", np.round(t3_losses, 6).tolist()) # -> [9.0, 3.24, 1.1664, 0.419904, 0.151165, 0.05442, 0.019591]
t3_predicted_final_error = -3.0 * t3_factor ** 6  # -> -0.139968
print("predicted final error:", round(t3_predicted_final_error, 6)) # -> -0.139968
assert abs(t3_errors[-1] - t3_predicted_final_error) < 1e-12

plt.figure(figsize=(4.6, 3.0))
plt.plot(range(t3_losses.size), t3_losses, marker="o", color="purple")
plt.yscale("log")
plt.title("Toy 3 · geometric loss decay")
plt.xlabel("iteration")
plt.ylabel("loss")
plt.show()

▶ What you'll see: the log-scale loss drops steadily because the error is multiplied by `0.6` each step.

### ✍️ Toy 4 · Step size controls stability

A learning rate below the quadratic stability limit contracts the path, while a too-large rate flips
sign and grows the error.

In [ ]:
import numpy as np                              # arrays and stability comparisons.

t4_rng = np.random.default_rng(0)               # -> seed 0
print("rng seed:", 0)                           # -> 0
t4_eta_good = 0.4                               # -> 0.4
print("stable eta:", t4_eta_good)               # -> 0.4
t4_eta_bad = 1.1                                # -> 1.1
print("unstable eta:", t4_eta_bad)              # -> 1.1
t4_good_path = np.array([0.0, 2.4, 2.88, 2.976, 2.9952, 2.99904])        # -> stable path
print("stable path:", np.round(t4_good_path, 6).tolist())               # -> [0.0, 2.4, 2.88, 2.976, 2.9952, 2.99904]
t4_bad_path = np.array([0.0, 6.6, -1.32, 8.184, -3.2208, 10.46496])      # -> unstable path
print("unstable path:", np.round(t4_bad_path, 6).tolist())              # -> [0.0, 6.6, -1.32, 8.184, -3.2208, 10.46496]
t4_good_loss = (t4_good_path - 3.0) ** 2       # -> [9.0, 0.36, 0.0144, 0.000576, 2.3e-05, 1e-06]
print("stable losses:", np.round(t4_good_loss, 6).tolist())             # -> [9.0, 0.36, 0.0144, 0.000576, 2.3e-05, 1e-06]
t4_bad_loss = (t4_bad_path - 3.0) ** 2         # -> [9.0, 12.96, 18.6624, 26.873856, 38.698353, 55.725628]
print("unstable losses:", np.round(t4_bad_loss, 6).tolist())            # -> [9.0, 12.96, 18.6624, 26.873856, 38.698353, 55.725628]
assert t4_good_loss[-1] < t4_good_loss[0] and t4_bad_loss[-1] > t4_bad_loss[0]

plt.figure(figsize=(4.8, 3.0))
plt.plot(t4_good_loss, marker="o", label="η=0.4")
plt.plot(t4_bad_loss, marker="s", label="η=1.1")
plt.yscale("log")
plt.title("Toy 4 · stable vs unstable η")
plt.xlabel("iteration")
plt.ylabel("loss")
plt.legend()
plt.show()

▶ What you'll see: the stable curve falls, while the large-step curve grows after overshooting.

### ✍️ Toy 5 · Vector gradients update every coordinate

In multiple dimensions, the gradient has one component per parameter. Steeper coordinates can move
farther even in the same update.

In [ ]:
import numpy as np                              # arrays and vector gradients.

t5_rng = np.random.default_rng(0)               # -> seed 0
print("rng seed:", 0)                           # -> 0
t5_w = np.array([3.0, 1.0])                     # -> [3.0, 1.0]
print("w:", t5_w.tolist())                      # -> [3.0, 1.0]
t5_grad = np.array([2.0 * (t5_w[0] - 1.0), 6.0 * (t5_w[1] + 1.0)])  # -> [4.0, 12.0]
print("gradient:", t5_grad.tolist())            # -> [4.0, 12.0]
t5_eta = 0.1                                    # -> 0.1
print("eta:", t5_eta)                           # -> 0.1
t5_step = -t5_eta * t5_grad                     # -> [-0.4, -1.2]
print("step:", t5_step.tolist())                # -> [-0.4, -1.2]
t5_next = t5_w + t5_step                        # -> [2.6, -0.2]
print("next w:", np.round(t5_next, 3).tolist()) # -> [2.6, -0.2]
t5_f0 = (t5_w[0] - 1.0) ** 2 + 3.0 * (t5_w[1] + 1.0) ** 2         # -> 16.0
print("f before:", t5_f0)                       # -> 16.0
t5_f1 = (t5_next[0] - 1.0) ** 2 + 3.0 * (t5_next[1] + 1.0) ** 2   # -> 4.48
print("f after:", round(float(t5_f1), 3))        # -> 4.48
t5_axis = np.linspace(-1.0, 3.0, 9)             # -> [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
print("plot axis:", t5_axis.tolist())           # -> [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
assert t5_f1 < t5_f0 and np.allclose(t5_next, [2.6, -0.2])

plt.figure(figsize=(4.4, 3.2))
plt.quiver([t5_w[0]], [t5_w[1]], [t5_step[0]], [t5_step[1]], angles="xy", scale_units="xy", scale=1, color="seagreen", label="GD step")
plt.scatter([t5_w[0], t5_next[0], 1.0], [t5_w[1], t5_next[1], -1.0], color=["crimson", "seagreen", "black"], zorder=3)
plt.xlim(-0.5, 3.5)
plt.ylim(-1.5, 1.5)
plt.title("Toy 5 · same-shaped vector step")
plt.xlabel("w0")
plt.ylabel("w1")
plt.legend()
plt.show()

▶ What you'll see: the second coordinate moves farther because its gradient component is larger.

### ✍️ Toy 6 · Regression gradients average examples

For one-parameter linear regression, each example contributes a gradient term. Their mean tells the
slope update which way to move.

In [ ]:
import numpy as np                              # arrays and regression gradients.

t6_rng = np.random.default_rng(0)               # -> seed 0
print("rng seed:", 0)                           # -> 0
t6_x = np.array([1.0, 2.0, 3.0, 4.0])           # -> [1.0, 2.0, 3.0, 4.0]
print("x:", t6_x.tolist())                      # -> [1.0, 2.0, 3.0, 4.0]
t6_y = 2.0 * t6_x                               # -> [2.0, 4.0, 6.0, 8.0]
print("y:", t6_y.tolist())                      # -> [2.0, 4.0, 6.0, 8.0]
t6_w0 = 0.0                                     # -> 0.0
print("initial w:", t6_w0)                      # -> 0.0
t6_pred0 = t6_w0 * t6_x                         # -> [0.0, 0.0, 0.0, 0.0]
print("initial predictions:", t6_pred0.tolist()) # -> [0.0, 0.0, 0.0, 0.0]
t6_resid = t6_pred0 - t6_y                      # -> [-2.0, -4.0, -6.0, -8.0]
print("residuals:", t6_resid.tolist())          # -> [-2.0, -4.0, -6.0, -8.0]
t6_terms = 2.0 * t6_x * t6_resid                # -> [-4.0, -16.0, -36.0, -64.0]
print("gradient terms:", t6_terms.tolist())     # -> [-4.0, -16.0, -36.0, -64.0]
t6_grad = float(t6_terms.mean())                # -> -30.0
print("mean gradient:", t6_grad)                # -> -30.0
t6_eta = 0.03                                   # -> 0.03
print("eta:", t6_eta)                           # -> 0.03
t6_w1 = t6_w0 - t6_eta * t6_grad                # -> 0.9
print("updated w:", round(t6_w1, 3))            # -> 0.9
t6_pred1 = t6_w1 * t6_x                         # -> [0.9, 1.8, 2.7, 3.6]
print("updated predictions:", np.round(t6_pred1, 3).tolist())   # -> [0.9, 1.8, 2.7, 3.6]
t6_loss0 = float(np.mean(t6_resid ** 2))        # -> 30.0
print("loss before:", t6_loss0)                 # -> 30.0
t6_loss1 = float(np.mean((t6_pred1 - t6_y) ** 2)) # -> 9.075
print("loss after:", round(t6_loss1, 3))         # -> 9.075
assert t6_loss1 < t6_loss0 and round(t6_w1, 3) == 0.9

plt.figure(figsize=(4.8, 3.0))
plt.scatter(t6_x, t6_y, color="black", label="data")
plt.plot(t6_x, t6_pred0, linestyle="--", color="crimson", label="before")
plt.plot(t6_x, t6_pred1, color="seagreen", label="after one step")
plt.title("Toy 6 · averaged regression gradient")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

▶ What you'll see: all examples push the slope upward, so one averaged step lowers the regression loss.


## 🛠️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

## 🟢 Basics (warm-up)

### Basic 1 — Evaluate a quadratic objective

**Goal.** Compute values of $f(x)=(x-3)^2$, because gradient descent needs an objective whose decrease we can inspect.

In [ ]:
x_b1 = np.array([0.0, 1.0, 3.0, 5.0])
f_b1 = (x_b1 - 3) ** 2
print("x values:", x_b1)
print("f(x):", f_b1)
assert f_b1[2] == 0.0

▶ What you'll see: the loss is smallest at `x=3` and grows away from it.

In [ ]:
grid_b1 = np.linspace(-1, 7, 200)
plt.figure(figsize=(4.6, 3))
plt.plot(grid_b1, (grid_b1 - 3) ** 2, color="navy")
plt.scatter(x_b1, f_b1, color="crimson", zorder=3)
plt.axvline(3, color="seagreen", linestyle="--", linewidth=1, label="minimum")
plt.title("Basic 1: quadratic objective values")
plt.xlabel("x"); plt.ylabel("f(x)"); plt.legend(); plt.show()

▶ What you'll see: the sampled points sit on a bowl whose lowest point is at `x=3`.

👀 Takeaway: optimization starts by naming the scalar quantity we want to make small.

### Basic 2 — Compute a derivative by formula

**Goal.** Use $f'(x)=2(x-3)$, because the gradient is the slope used by descent.

In [ ]:
x_b2 = 0.0
grad_b2 = 2 * (x_b2 - 3)
print("x:", x_b2, "gradient:", grad_b2)
assert grad_b2 == -6.0

▶ What you'll see: the slope at zero is negative.

In [ ]:
grid_b2 = np.linspace(-1, 5, 200)
loss_grid_b2 = (grid_b2 - 3) ** 2
tangent_b2 = (x_b2 - 3) ** 2 + grad_b2 * (grid_b2 - x_b2)
plt.figure(figsize=(4.6, 3))
plt.plot(grid_b2, loss_grid_b2, color="navy", label="f(x)")
plt.plot(grid_b2, tangent_b2, "--", color="crimson", label="tangent at x=0")
plt.scatter([x_b2], [(x_b2 - 3) ** 2], color="black", zorder=3)
plt.title("Basic 2: derivative as local slope")
plt.xlabel("x"); plt.ylabel("value"); plt.ylim(-2, 11); plt.legend(); plt.show()

▶ What you'll see: the tangent line slopes downward as `x` increases, matching the negative derivative.

👀 Takeaway: the sign of the gradient tells which direction is locally uphill.

### Basic 3 — Move opposite the derivative

**Goal.** Take one update $x\leftarrow x-\eta f'(x)$, because descent negates the uphill direction.

In [ ]:
x_b3 = 0.0
eta_b3 = 0.1
grad_b3 = 2 * (x_b3 - 3)
x_next_b3 = x_b3 - eta_b3 * grad_b3
print("next x:", x_next_b3)
assert round(x_next_b3, 3) == 0.6

▶ What you'll see: subtracting a negative gradient moves `x` to the right.

In [ ]:
grid_b3 = np.linspace(-1, 5, 200)
plt.figure(figsize=(4.6, 3))
plt.plot(grid_b3, (grid_b3 - 3) ** 2, color="navy")
plt.scatter([x_b3, x_next_b3], [(x_b3 - 3) ** 2, (x_next_b3 - 3) ** 2],
            color=["crimson", "seagreen"], zorder=3)
plt.annotate("", xy=(x_next_b3, (x_next_b3 - 3) ** 2), xytext=(x_b3, (x_b3 - 3) ** 2),
             arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))
plt.title("Basic 3: one step opposite the gradient")
plt.xlabel("x"); plt.ylabel("f(x)"); plt.show()

▶ What you'll see: the arrow moves from the starting point to a lower point on the bowl.

👀 Takeaway: gradient descent is “current point minus step size times gradient.”

### Basic 4 — Verify the loss decreased

**Goal.** Compare loss before and after one step, because the update should improve the objective when the step is safe.

In [ ]:
x0_b4 = 0.0
x1_b4 = 0.6
loss0_b4 = (x0_b4 - 3) ** 2
loss1_b4 = (x1_b4 - 3) ** 2
print("loss before:", loss0_b4, "loss after:", loss1_b4)
assert loss1_b4 < loss0_b4 and round(loss1_b4, 2) == 5.76

▶ What you'll see: the loss falls from `9.0` to `5.76`.

In [ ]:
plt.figure(figsize=(4.2, 3))
plt.bar(["before", "after"], [loss0_b4, loss1_b4], color=["crimson", "seagreen"])
plt.title("Basic 4: one safe step lowers loss")
plt.ylabel("f(x)"); plt.show()

▶ What you'll see: the after-step bar is shorter, confirming objective decrease.

👀 Takeaway: one good gradient step should be checked by actual objective decrease.

### Basic 5 — Run five gradient steps

**Goal.** Repeat the update a few times, because optimization is accumulated local improvement.

In [ ]:
eta_b5 = 0.2
xs_b5 = [0.0]
for step_b5 in range(5):
    grad_b5 = 2 * (xs_b5[-1] - 3)
    xs_b5.append(xs_b5[-1] - eta_b5 * grad_b5)
xs_b5 = np.array(xs_b5)
print("path:", np.round(xs_b5, 3))
assert np.all(np.diff((xs_b5 - 3) ** 2) < 0)

▶ What you'll see: each iterate gets closer to `3`.

In [ ]:
grid_b5 = np.linspace(-0.5, 3.5, 200)
plt.figure(figsize=(4.8, 3))
plt.plot(grid_b5, (grid_b5 - 3) ** 2, color="navy")
plt.plot(xs_b5, (xs_b5 - 3) ** 2, marker="o", color="crimson")
plt.title("Basic 5: five-step descent path")
plt.xlabel("x"); plt.ylabel("f(x)"); plt.show()

▶ What you'll see: the points move down the bowl and bunch closer to the minimum.

👀 Takeaway: safe repeated steps contract the distance to the minimizer.

### Basic 6 — Plot the path on the bowl

**Goal.** Visualize iterates on the objective, because plots reveal whether descent is moving sensibly.

In [ ]:
grid_b6 = np.linspace(-1, 5, 200)
loss_grid_b6 = (grid_b6 - 3) ** 2
xs_b6 = np.array([0.0, 1.2, 1.92, 2.352, 2.6112])
plt.figure(figsize=(4.6, 3))
plt.plot(grid_b6, loss_grid_b6, color="navy")
plt.scatter(xs_b6, (xs_b6 - 3) ** 2, color="crimson")
plt.title("Basic 6: iterates on the quadratic")
plt.xlabel("x"); plt.ylabel("f(x)"); plt.show()

▶ What you'll see: the red points walk down the bowl toward the minimum.

👀 Takeaway: a trajectory plot is a simple debugging tool for descent behavior.

### Basic 7 — Compute the geometric error factor

**Goal.** Derive the contraction factor for the quadratic, because it predicts convergence speed.

In [ ]:
eta_b7 = 0.2
factor_b7 = 1 - 2 * eta_b7
error0_b7 = -3.0
error5_b7 = (factor_b7 ** 5) * error0_b7
print("factor:", factor_b7, "error after 5 steps:", round(error5_b7, 4))
assert round(factor_b7, 3) == 0.6

▶ What you'll see: with `η=0.2`, the error keeps 60% of its previous value each step.

In [ ]:
steps_b7 = np.arange(0, 8)
errors_b7 = np.abs((factor_b7 ** steps_b7) * error0_b7)
plt.figure(figsize=(4.6, 3))
plt.plot(steps_b7, errors_b7, marker="o", color="purple")
plt.yscale("log")
plt.title("Basic 7: geometric error decay")
plt.xlabel("step"); plt.ylabel("|x - 3|, log scale"); plt.show()

▶ What you'll see: each point is 60% of the previous one, so the log-scale curve falls steadily.

👀 Takeaway: on a quadratic, step size directly controls the contraction factor.

### Basic 8 — Show overshooting with a large step

**Goal.** Try a too-large learning rate, because correct directions can still diverge.

In [ ]:
eta_b8 = 1.1
xs_b8 = [0.0]
for step_b8 in range(4):
    grad_b8 = 2 * (xs_b8[-1] - 3)
    xs_b8.append(xs_b8[-1] - eta_b8 * grad_b8)
xs_b8 = np.array(xs_b8)
print("large-step path:", np.round(xs_b8, 3))
assert abs(xs_b8[-1] - 3) > abs(xs_b8[0] - 3)

▶ What you'll see: the sequence crosses back and forth with growing amplitude.

In [ ]:
grid_b8 = np.linspace(min(xs_b8) - 1, max(xs_b8) + 1, 300)
plt.figure(figsize=(5, 3))
plt.plot(grid_b8, (grid_b8 - 3) ** 2, color="navy")
plt.plot(xs_b8, (xs_b8 - 3) ** 2, marker="o", color="crimson")
plt.axvline(3, color="seagreen", linestyle="--", linewidth=1)
plt.title("Basic 8: large-step overshooting")
plt.xlabel("x"); plt.ylabel("f(x)"); plt.show()

▶ What you'll see: the path alternates across the minimizer while climbing to larger losses.

👀 Takeaway: learning rate is a stability choice, not just a speed choice.

### Basic 9 — Update a two-parameter vector

**Goal.** Apply gradient descent to a vector parameter, because ML models usually have many parameters.

In [ ]:
w_b9 = np.array([-2.0, 1.0])
grad_b9 = np.array([2 * (w_b9[0] - 1), 8 * (w_b9[1] + 2)])
eta_b9 = 0.1
w_next_b9 = w_b9 - eta_b9 * grad_b9
print("gradient:", grad_b9, "next w:", w_next_b9)
assert np.allclose(w_next_b9, [-1.4, -1.4])

▶ What you'll see: each coordinate is updated by its own partial derivative.

In [ ]:
x_b9 = np.linspace(-3, 2, 80)
y_b9 = np.linspace(-3, 2, 80)
X_b9, Y_b9 = np.meshgrid(x_b9, y_b9)
Z_b9 = (X_b9 - 1) ** 2 + 4 * (Y_b9 + 2) ** 2
plt.figure(figsize=(4.8, 3.6))
plt.contour(X_b9, Y_b9, Z_b9, levels=18, cmap="viridis")
plt.scatter([w_b9[0], w_next_b9[0], 1], [w_b9[1], w_next_b9[1], -2],
            color=["crimson", "seagreen", "black"], zorder=3)
plt.quiver(w_b9[0], w_b9[1], w_next_b9[0] - w_b9[0], w_next_b9[1] - w_b9[1],
           angles="xy", scale_units="xy", scale=1, color="gray")
plt.title("Basic 9: vector update on contours")
plt.xlabel("w0"); plt.ylabel("w1"); plt.show()

▶ What you'll see: the quiver arrow shows the two-coordinate step moving toward the bowl center.

👀 Takeaway: the gradient has the same shape as the parameter vector.

### Basic 10 — One regression-gradient step

**Goal.** Compute the first linear-regression update, because the same descent rule trains models.

In [ ]:
X_b10 = np.array([1.0, 2.0, 3.0])
y_b10 = 2 * X_b10
w_b10 = 0.0
grad_b10 = np.mean(2 * X_b10 * (w_b10 * X_b10 - y_b10))
w_next_b10 = w_b10 - 0.05 * grad_b10
print("gradient:", round(grad_b10, 3), "next w:", round(w_next_b10, 3))
assert round(w_next_b10, 3) == 0.933

▶ What you'll see: the slope jumps upward because predictions are too small.

In [ ]:
plt.figure(figsize=(4.8, 3.2))
plt.scatter(X_b10, y_b10, color="black", label="data")
plt.plot(X_b10, w_b10 * X_b10, "--", color="crimson", label="before")
plt.plot(X_b10, w_next_b10 * X_b10, color="seagreen", label="after one step")
plt.title("Basic 10: regression line after one update")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

▶ What you'll see: the fitted line rotates upward toward the data after the gradient step.

👀 Takeaway: regression training is gradient descent on average squared prediction error.

## 🟡 Easy

### Easy 1 — Plot a full convergence curve

**Goal.** Track loss over many iterations, because convergence should be visible as a falling curve.

In [ ]:
eta_e1 = 0.2
x_e1 = 0.0
losses_e1 = []
for step_e1 in range(25):
    losses_e1.append((x_e1 - 3) ** 2)
    grad_e1 = 2 * (x_e1 - 3)
    x_e1 = x_e1 - eta_e1 * grad_e1
print("final x:", round(x_e1, 6), "final loss:", round((x_e1 - 3) ** 2, 10))
assert losses_e1[-1] < losses_e1[0]

▶ What you'll see: the final point is extremely close to the minimizer.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(losses_e1, marker="o", color="teal")
plt.yscale("log")
plt.title("Easy 1: gradient descent convergence")
plt.xlabel("iteration"); plt.ylabel("loss, log scale"); plt.show()

▶ What you'll see: a smoothly decreasing loss curve.

👀 Takeaway: convergence diagnostics turn repeated updates into inspectable evidence.

### Easy 2 — Compare several learning rates

**Goal.** Sweep step sizes, because the same gradient formula can be slow, fast, or unstable.

In [ ]:
etas_e2 = np.array([0.05, 0.2, 0.8, 1.1])
final_losses_e2 = []
paths_e2 = []
for eta_e2 in etas_e2:
    x_e2 = 0.0
    path_e2 = []
    for step_e2 in range(15):
        path_e2.append((x_e2 - 3) ** 2)
        x_e2 = x_e2 - eta_e2 * 2 * (x_e2 - 3)
    paths_e2.append(path_e2)
    final_losses_e2.append((x_e2 - 3) ** 2)
print("final losses:", np.round(final_losses_e2, 3))
assert final_losses_e2[-1] > final_losses_e2[0]

▶ What you'll see: moderate learning rates beat tiny ones, while `1.1` is unstable.

In [ ]:
plt.figure(figsize=(5, 3))
for eta_e2, path_e2 in zip(etas_e2, paths_e2):
    plt.plot(path_e2, label=f"η={eta_e2}")
plt.yscale("log"); plt.title("Easy 2: learning-rate sweep")
plt.xlabel("iteration"); plt.ylabel("loss"); plt.legend(); plt.show()

▶ What you'll see: the stable curves fall; the too-large curve rises.

👀 Takeaway: learning rate is the main knob controlling progress versus overshoot.

### Easy 3 — Train one-parameter linear regression

**Goal.** Fit $\hat y=wx$ by gradient descent, because model training uses the same update on data loss.

In [ ]:
X_e3 = np.array([1.0, 2.0, 3.0, 4.0])
y_e3 = 2.0 * X_e3
w_e3 = 0.0
losses_e3 = []
for step_e3 in range(60):
    pred_e3 = w_e3 * X_e3
    losses_e3.append(np.mean((pred_e3 - y_e3) ** 2))
    grad_e3 = np.mean(2 * X_e3 * (pred_e3 - y_e3))
    w_e3 = w_e3 - 0.03 * grad_e3
print("learned w:", round(w_e3, 4), "start/end loss:", round(losses_e3[0], 3), round(losses_e3[-1], 6))
assert abs(w_e3 - 2.0) < 0.01

▶ What you'll see: the learned slope approaches the true value `2.0`.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.plot(losses_e3, color="purple")
plt.title("Easy 3: regression loss falls")
plt.xlabel("step"); plt.ylabel("MSE"); plt.show()

▶ What you'll see: MSE decreases toward zero as the line fits the data.

👀 Takeaway: gradients average per-example residual information into a parameter update.

### Easy 4 — See feature scaling change gradient size

**Goal.** Compare gradients before and after rescaling inputs, because feature scale changes safe learning rates.

In [ ]:
X_raw_e4 = np.array([10.0, 20.0, 30.0])
y_raw_e4 = 2 * X_raw_e4
w_e4 = 0.0
grad_raw_e4 = np.mean(2 * X_raw_e4 * (w_e4 * X_raw_e4 - y_raw_e4))
X_scaled_e4 = X_raw_e4 / 10.0
y_scaled_e4 = 2 * X_scaled_e4
grad_scaled_e4 = np.mean(2 * X_scaled_e4 * (w_e4 * X_scaled_e4 - y_scaled_e4))
print("raw gradient:", round(grad_raw_e4, 3), "scaled gradient:", round(grad_scaled_e4, 3))
assert abs(grad_raw_e4 / grad_scaled_e4 - 100) < 1e-9

▶ What you'll see: scaling `x` by 10 changes this gradient magnitude by 100.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["raw", "scaled"], [abs(grad_raw_e4), abs(grad_scaled_e4)], color=["crimson", "seagreen"])
plt.yscale("log")
plt.title("Easy 4: scale changes gradient magnitude")
plt.ylabel("|gradient|, log scale"); plt.show()

▶ What you'll see: the raw feature produces a much larger gradient.

👀 Takeaway: rescaling inputs can turn an unsafe learning rate into a safe one.

### Easy 5 — Use a gradient-norm stopping rule

**Goal.** Stop when the gradient is small, because near a smooth minimum the slope should approach zero.

In [ ]:
x_e5 = 0.0
eta_e5 = 0.2
tol_e5 = 1e-3
steps_e5 = 0
while abs(2 * (x_e5 - 3)) > tol_e5 and steps_e5 < 1000:
    x_e5 = x_e5 - eta_e5 * 2 * (x_e5 - 3)
    steps_e5 += 1
print("steps:", steps_e5, "x:", round(x_e5, 6), "gradient:", round(2 * (x_e5 - 3), 8))
assert abs(2 * (x_e5 - 3)) <= tol_e5

▶ What you'll see: the loop stops once the local slope is tiny.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["distance to optimum", "|gradient|"], [abs(x_e5 - 3), abs(2 * (x_e5 - 3))], color="teal")
plt.title("Easy 5: stopping quantities")
plt.show()

▶ What you'll see: both the distance and gradient norm are very small on this convex quadratic.

👀 Takeaway: small gradients are useful stopping signals, especially in convex smooth problems.

## 🔴 Advanced

### Advanced 1 — Descend an ill-conditioned bowl

**Goal.** Optimize an elliptical quadratic, because different curvatures make one learning rate behave unevenly across coordinates.

In [ ]:
w_a1 = np.array([5.0, 5.0])
eta_a1 = 0.08
path_a1 = [w_a1.copy()]
for step_a1 in range(40):
    grad_a1 = np.array([2 * w_a1[0], 20 * w_a1[1]])
    w_a1 = w_a1 - eta_a1 * grad_a1
    path_a1.append(w_a1.copy())
path_a1 = np.array(path_a1)
loss_a1 = path_a1[:, 0] ** 2 + 10 * path_a1[:, 1] ** 2
print("start/end loss:", round(loss_a1[0], 3), round(loss_a1[-1], 6))
assert loss_a1[-1] < loss_a1[0]

▶ What you'll see: the loss decreases, but the steep coordinate can oscillate.

In [ ]:
plt.figure(figsize=(4.8, 3.5))
plt.plot(path_a1[:, 0], path_a1[:, 1], marker="o", markersize=3, color="crimson")
plt.title("Advanced 1: path on ill-conditioned bowl")
plt.xlabel("w0"); plt.ylabel("w1"); plt.show()

▶ What you'll see: the path zigzags because one coordinate has much larger curvature.

👀 Takeaway: ill-conditioning makes one global step size a compromise across coordinates.

### Advanced 2 — Compare batch and stochastic gradients

**Goal.** Contrast full-data and one-example gradients, because stochastic optimization replaces exact gradients with noisy estimates.

In [ ]:
X_a2 = np.array([1.0, 2.0, 3.0, 4.0])
y_a2 = 2 * X_a2
w_batch_a2 = 0.0
w_sgd_a2 = 0.0
eta_a2 = 0.03
batch_losses_a2 = []
sgd_losses_a2 = []
for step_a2 in range(80):
    pred_batch_a2 = w_batch_a2 * X_a2
    grad_batch_a2 = np.mean(2 * X_a2 * (pred_batch_a2 - y_a2))
    w_batch_a2 -= eta_a2 * grad_batch_a2
    i_a2 = step_a2 % len(X_a2)
    grad_sgd_a2 = 2 * X_a2[i_a2] * (w_sgd_a2 * X_a2[i_a2] - y_a2[i_a2])
    w_sgd_a2 -= eta_a2 * grad_sgd_a2
    batch_losses_a2.append(np.mean((w_batch_a2 * X_a2 - y_a2) ** 2))
    sgd_losses_a2.append(np.mean((w_sgd_a2 * X_a2 - y_a2) ** 2))
print("batch w:", round(w_batch_a2, 3), "sgd w:", round(w_sgd_a2, 3))
assert abs(w_batch_a2 - 2) < 0.01 and abs(w_sgd_a2 - 2) < 0.05

▶ What you'll see: both approaches learn the slope, but SGD takes noisier steps.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(batch_losses_a2, label="batch")
plt.plot(sgd_losses_a2, label="stochastic", alpha=0.8)
plt.yscale("log")
plt.title("Advanced 2: exact vs noisy gradients")
plt.xlabel("update"); plt.ylabel("MSE"); plt.legend(); plt.show()

▶ What you'll see: the stochastic curve is less smooth but still trends downward.

👀 Takeaway: noisy gradients can target the same optimum while reducing per-step cost.

### Advanced 3 — Use momentum to smooth zigzags

**Goal.** Add a velocity term, because momentum accumulates consistent gradient directions and damps alternating ones.

In [ ]:
w_plain_a3 = np.array([5.0, 5.0])
w_mom_a3 = np.array([5.0, 5.0])
v_a3 = np.zeros(2)
eta_a3 = 0.08
beta_a3 = 0.8
loss_plain_a3 = []
loss_mom_a3 = []
for step_a3 in range(60):
    grad_plain_a3 = np.array([2 * w_plain_a3[0], 20 * w_plain_a3[1]])
    w_plain_a3 -= eta_a3 * grad_plain_a3
    grad_mom_a3 = np.array([2 * w_mom_a3[0], 20 * w_mom_a3[1]])
    v_a3 = beta_a3 * v_a3 + grad_mom_a3
    w_mom_a3 -= eta_a3 * v_a3
    loss_plain_a3.append(w_plain_a3[0] ** 2 + 10 * w_plain_a3[1] ** 2)
    loss_mom_a3.append(w_mom_a3[0] ** 2 + 10 * w_mom_a3[1] ** 2)
print("final plain/momentum loss:", round(loss_plain_a3[-1], 5), round(loss_mom_a3[-1], 5))
assert np.isfinite(loss_mom_a3[-1])

▶ What you'll see: momentum changes the trajectory by using accumulated past gradients.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(loss_plain_a3, label="plain GD")
plt.plot(loss_mom_a3, label="momentum")
plt.yscale("log")
plt.title("Advanced 3: momentum changes convergence")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()

▶ What you'll see: momentum can accelerate broad directions but may oscillate if too aggressive.

👀 Takeaway: optimizer variants still rely on gradients, but modify how steps are accumulated.

### Advanced 4 — Detect a saddle with a small gradient

**Goal.** Inspect $f(x,y)=x^2-y^2$, because a zero gradient is not always a minimum.

In [ ]:
point_a4 = np.array([0.0, 0.0])
grad_a4 = np.array([2 * point_a4[0], -2 * point_a4[1]])
H_a4 = np.array([[2.0, 0.0], [0.0, -2.0]])
eigs_a4 = np.linalg.eigvalsh(H_a4)
print("gradient:", grad_a4, "Hessian eigenvalues:", eigs_a4)
assert np.allclose(grad_a4, [0, 0]) and eigs_a4[0] < 0 < eigs_a4[1]

▶ What you'll see: the gradient is zero, but curvature is positive in one direction and negative in another.

In [ ]:
x_a4 = np.linspace(-2, 2, 80)
y_a4 = np.linspace(-2, 2, 80)
X_a4, Y_a4 = np.meshgrid(x_a4, y_a4)
Z_a4 = X_a4 ** 2 - Y_a4 ** 2
plt.figure(figsize=(4.8, 3.5))
plt.contour(X_a4, Y_a4, Z_a4, levels=21, cmap="coolwarm")
plt.scatter([0], [0], color="black")
plt.title("Advanced 4: zero-gradient saddle")
plt.xlabel("x"); plt.ylabel("y"); plt.show()

▶ What you'll see: contours bend upward along one axis and downward along the other.

👀 Takeaway: small gradients alone do not prove optimality in nonconvex problems.

### Advanced 5 — Gradient descent with standardized features

**Goal.** Fit two-feature regression before and after standardization, because scaling improves conditioning and learning-rate safety.

In [ ]:
X_raw_a5 = np.array([[1.0, 100.0], [2.0, 200.0], [3.0, 300.0], [4.0, 400.0]])
y_a5 = np.array([3.0, 6.0, 9.0, 12.0])
mean_a5 = X_raw_a5.mean(axis=0)
std_a5 = X_raw_a5.std(axis=0)
X_a5 = (X_raw_a5 - mean_a5) / std_a5
w_a5 = np.zeros(2)
b_a5 = 0.0
losses_a5 = []
for step_a5 in range(100):
    pred_a5 = X_a5 @ w_a5 + b_a5
    err_a5 = pred_a5 - y_a5
    losses_a5.append(np.mean(err_a5 ** 2))
    grad_w_a5 = (2 / len(y_a5)) * (X_a5.T @ err_a5)
    grad_b_a5 = 2 * np.mean(err_a5)
    w_a5 -= 0.1 * grad_w_a5
    b_a5 -= 0.1 * grad_b_a5
print("final loss:", round(losses_a5[-1], 8), "bias:", round(b_a5, 3), "weights:", np.round(w_a5, 3))
assert losses_a5[-1] < 1e-6

▶ What you'll see: standardized features allow a simple learning rate to drive loss nearly to zero.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(losses_a5, color="seagreen")
plt.yscale("log")
plt.title("Advanced 5: standardized-feature descent")
plt.xlabel("step"); plt.ylabel("MSE"); plt.show()

▶ What you'll see: the loss falls smoothly despite the raw features living on very different scales.

👀 Takeaway: feature standardization improves the geometry that gradient descent sees.